In [0]:

# Leer resultados del pipeline
tabla_limpias = (
    "workspace.ingenieria_datos_juanpa."
    "pipeline_ventas_limpias"
)

tabla_resumen = (
    "workspace.ingenieria_datos_juanpa."
    "pipeline_ventas_por_categoria"
)

df_limpias = spark.table(tabla_limpias)
df_resumen = spark.table(tabla_resumen)

print("Ventas limpias:", df_limpias.count())
print("Categorías:", df_resumen.count())

display(df_resumen.orderBy("categoria"))


Ventas limpias: 15
Categorías: 4


categoria,total_vendido
Accesorios,1125000
Hogar,1910000
Papelería,254000
Tecnología,8205000


In [0]:

# Agregar una venta válida a Bronze
# MERGE evita insertarla dos veces por accidente

spark.sql("""
MERGE INTO workspace.ingenieria_datos_juanpa.ventas_bronze AS destino
USING (
    SELECT
        19 AS id_venta,
        DATE '2026-11-13' AS fecha,
        'Resma de papel' AS producto,
        'Papelería' AS categoria,
        2 AS cantidad,
        CAST(100000 AS BIGINT) AS precio_unitario
) AS nueva
ON destino.id_venta = nueva.id_venta

WHEN NOT MATCHED THEN INSERT (
    id_venta,
    fecha,
    producto,
    categoria,
    cantidad,
    precio_unitario
)
VALUES (
    nueva.id_venta,
    nueva.fecha,
    nueva.producto,
    nueva.categoria,
    nueva.cantidad,
    nueva.precio_unitario
)
""")

print("Registros Bronze:", spark.table(
    "workspace.ingenieria_datos_juanpa.ventas_bronze"
).count())


Registros Bronze: 19


In [0]:

from pyspark.sql import functions as F

# Leer las vistas materializadas actualizadas
df_limpias = spark.table(
    "workspace.ingenieria_datos_juanpa.pipeline_ventas_limpias"
)

df_resumen = spark.table(
    "workspace.ingenieria_datos_juanpa.pipeline_ventas_por_categoria"
)

print("Registros limpios:", df_limpias.count())

display(df_resumen.orderBy("categoria"))

# Consultar total de todas las categorías
df_resumen.agg(
    F.sum("total_vendido").alias("total_general")
).show()


Registros limpios: 16


categoria,total_vendido
Accesorios,1125000
Hogar,1910000
Papelería,454000
Tecnología,8205000


+-------------+
|total_general|
+-------------+
|     11694000|
+-------------+



In [0]:

from pyspark.sql import functions as F

# Resultados anteriores a la nueva venta
datos_anteriores = [
    ("Accesorios", 1125000),
    ("Hogar", 1910000),
    ("Papelería", 254000),
    ("Tecnología", 8205000)
]

# Crear DataFrame con los valores anteriores
df_antes = spark.createDataFrame(
    datos_anteriores,
    ["categoria", "total_antes"]
)

# Leer los resultados actuales del pipeline
df_despues = spark.table(
    "workspace.ingenieria_datos_juanpa.pipeline_ventas_por_categoria"
).select(
    "categoria",
    F.col("total_vendido").alias("total_despues")
)

# Comparar los resultados
df_comparacion = df_antes.join(
    df_despues,
    on="categoria",
    how="full"
).fillna(0)

df_comparacion = df_comparacion.withColumn(
    "diferencia",
    F.col("total_despues") - F.col("total_antes")
)

# Mostrar resultados de la comparación
display(df_comparacion.orderBy("categoria"))

# Calcular totales generales
df_comparacion.agg(
    F.sum("total_antes").alias("total_antes"),
    F.sum("total_despues").alias("total_despues"),
    F.sum("diferencia").alias("diferencia_total")
).show()


categoria,total_antes,total_despues,diferencia
Accesorios,1125000,1125000,0
Hogar,1910000,1910000,0
Papelería,254000,454000,200000
Tecnología,8205000,8205000,0


+-----------+-------------+----------------+
|total_antes|total_despues|diferencia_total|
+-----------+-------------+----------------+
|   11494000|     11694000|          200000|
+-----------+-------------+----------------+

